# Pipeline ETL de Ventas Retail: Consolidación y Limpieza de Datos Multi-Fuente

**Categoría de mercado:** Productos de limpieza para el hogar (canal autoservicio, México)

**Rol:** Diseño y desarrollo end-to-end del pipeline (individual)

**Stack:** Python · pandas · NumPy

## Contexto

Una empresa de bienes de consumo con presencia en el canal de autoservicio en México requiere
consolidar su información de ventas, dispersa en cinco fuentes independientes (ventas semanales,
catálogo de productos, categorías, segmentos de mercado y calendario comercial), en un único
dataset limpio y analítico, listo para alimentar reportes, dashboards y modelos de segmentación.

Los datos provienen de un esquema en estrella: una tabla de hechos (`FACT_SALES`, ~122K registros,
2022-2023) y cuatro tablas de dimensión (`DIM_PRODUCT`, `DIM_CATEGORY`, `DIM_SEGMENT`,
`DIM_CALENDAR`).

## Objetivo

Construir un pipeline ETL reproducible que:

1. Cargue y valide la calidad de las cinco fuentes.
2. Corrija inconsistencias de formato, texto y valores nulos.
3. Consolide las tablas en un único dataset a nivel de venta semanal por producto y región.
4. Genere variables derivadas (temporales, de precio, de desempeño y de ranking) que soporten
   análisis exploratorio y modelos posteriores (EDA, clustering, etc).
5. Detecte y corrija errores estructurales de los datos de origen que, de no tratarse, sesgarían
   cualquier análisis posterior.

## Resultado

Un dataset consolidado de 122,002 filas × 47 columnas, sin nulos ni duplicados, con banderas
explícitas para evitar el doble conteo regional y variables listas para segmentación y modelado.

---

# 1. Importación de librerías y carga de las fuentes de datos

Se cargan las cinco tablas del esquema en estrella: la tabla de hechos `FACT_SALES`, y las cuatro tablas de dimensión`DIM_PRODUCT`, `DIM_CATEGORY`, `DIM_SEGMENT`, `DIM_CALENDAR`

In [40]:
import pandas as pd
import numpy as np

df_category = pd.read_csv('DIM_CATEGORY.csv')
df_sales = pd.read_csv('FACT_SALES.csv')
df_calendar = pd.read_excel('DIM_CALENDAR.xlsx')
df_product = pd.read_excel('DIM_PRODUCT.xlsx')
df_segment = pd.read_excel('DIM_SEGMENT.xlsx')

print('Dataframes:\n')

for nombre, df in [('DIM_CATEGORY', df_category), ('DIM_PRODUCT', df_product), ('DIM_SEGMENT', df_segment), 
                   ('DIM_CALENDAR', df_calendar), ('FACT_SALES', df_sales)]:
    print(f"{nombre:15} {df.shape[0]:>8,} filas x {df.shape[1]} columnas")

Dataframes:

DIM_CATEGORY           5 filas x 2 columnas
DIM_PRODUCT          505 filas x 9 columnas
DIM_SEGMENT           53 filas x 6 columnas
DIM_CALENDAR         156 filas x 5 columnas
FACT_SALES       122,002 filas x 6 columnas


# 2. Exploración inicial

Antes de limpiar, se revisa la estructura, tipos de datos, nulos y duplicados de cada tabla para entender qué tratamiento necesita cada una.

In [41]:
def revisar_df(df, nombre):
    print(f"\n{'='*90}\n{nombre}\n{'='*90}")
    print(f"• Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"• Columnas: {df.columns.tolist()}")
    print(f"• Valores nulos por columna: \n{df.isnull().sum()}")
    print(f"• Nulos totales: {df.isnull().sum().sum()}")
    print(f"• Filas duplicadas: {df.duplicated().sum()}")
    print(f"• Valores únicos por columna: \n{df.nunique()}")
    print(f"• Tipos de datos por columna: \n{df.dtypes}")
    print(f"• Estadísticas numéricas: \n{df.describe()}")
    print(f"• Estadísticas de texto: \n{df.describe(include='string')}")
    
    if len(df) > 10:
        print(f"\n• Muestra de 5 filas aleatorias: \n{df.sample(5)}")
    else:
        print(f"\n• DataFrame completo: \n{df.to_string()}")

for nombre, df in [('DIM_CATEGORY', df_category), ('DIM_PRODUCT', df_product), ('DIM_SEGMENT', df_segment), 
                   ('DIM_CALENDAR', df_calendar), ('FACT_SALES', df_sales)]:
    revisar_df(df, nombre)


DIM_CATEGORY
• Dimensiones: 5 filas x 2 columnas
• Columnas: ['ID_CATEGORY', 'CATEGORY']
• Valores nulos por columna: 
ID_CATEGORY    0
CATEGORY       0
dtype: int64
• Nulos totales: 0
• Filas duplicadas: 0
• Valores únicos por columna: 
ID_CATEGORY    5
CATEGORY       5
dtype: int64
• Tipos de datos por columna: 
ID_CATEGORY    int64
CATEGORY         str
dtype: object
• Estadísticas numéricas: 
       ID_CATEGORY
count     5.000000
mean      3.000000
std       1.581139
min       1.000000
25%       2.000000
50%       3.000000
75%       4.000000
max       5.000000
• Estadísticas de texto: 
                              CATEGORY
count                                5
unique                               5
top     FABRIC TREATMENT and SANIT\r\n
freq                                 1

• DataFrame completo: 
   ID_CATEGORY                        CATEGORY
0            1  FABRIC TREATMENT and SANIT\r\n
1            2                       AIR CARE 
2            3                    LAVAVAJIL

### Hallazgos de la exploración:

- `DIM_PRODUCT` tiene 14 nulos y `DIM_SEGMENT` 1 nulo, ambos en columnas de texto.
- `DIM_CATEGORY` contiene un carácter de salto de línea (`\r\n`) incrustado en un valor de texto, un problema típico de exportación desde Excel/SQL que puede romper joins si no se limpia bien.
- Al revisar manualmente el catálogo de productos se detectaron 11 descripciones que comienzan con un "9" que no pertenece al nombre del producto (p. ej. `9TIDE ADIVITO QUITAMANCHAS 220 GR` en vez de `TIDE ADIVITO QUITAMANCHAS 220 GR`). Es un typo de captura/carga, no un dato válido.
- `FACT_SALES` no tiene nulos, pero contiene una fila `TOTAL AUTOS SCANNING MEXICO` dentro de la columna `REGION`, que parece una región más pero es en realidad un consolidado nacional, no una región independiente.

# 3. Limpieza y estandarización

Se aplican dos tipos de corrección:

1. **Corrección puntual del typo "9"** en `ITEM_DESCRIPTION` de `DIM_PRODUCT`. Solo afecta a las 11 filas donde el "9" antecede directamente a una letra al inicio del texto (patrón que no aparece en ninguna descripción legítima del catálogo, verificado antes de aplicar el fix).
2. **Limpieza general** aplicada a las cinco tablas: estandarización de texto (espacios, formato de título), tratamiento de nulos con una estrategia distinta según la dependencia de cada columna, y eliminación de duplicados.

In [42]:
# Corrección del typo "9" en ITEM_DESCRIPTION (antes de estandarizar texto)
mask_typo9 = df_product['ITEM_DESCRIPTION'].astype(str).str.match(r'^9[A-Za-z]')
print(f"• Descripciones con '9' inicial espurio: {mask_typo9.sum()}\n")
df_product.loc[mask_typo9, 'ITEM_DESCRIPTION'] = (
    df_product.loc[mask_typo9, 'ITEM_DESCRIPTION'].str.replace(r'^9', '', regex=True)
)
print("• Ejemplos corregidos:")
print(df_product.loc[mask_typo9, 'ITEM_DESCRIPTION'].head(5).tolist())

• Descripciones con '9' inicial espurio: 11

• Ejemplos corregidos:
['TIDE LIPIADOR DE ROPA QUITAMANCHAS PLUMON 10 ML IMP 0037000018704', 'TIDE ADIVITO QUITAMANCHAS 220 GR 0037000213215', 'TIDE BOOST VIVID WHITE + BRIGHT BOL 18CAP = 400GR IMP 0037000830870', 'VALUE CLORO SPECIAL BOT PLAST 3780ML 0041380309109', 'ARCTIC WHITE FRESCO BOT.PLAS 3.79ML MP 0041596020959']


In [43]:
def limpiar_df(df, nombre):
    '''Estandariza texto, trata nulos según dependencia de columnas y elimina duplicados.'''
    filas_ini, nulos_ini, dup_ini = len(df), df.isnull().sum().sum(), df.duplicated().sum()

    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = df[col].str.strip().str.title().str.replace(r'\s+', ' ', regex=True)

    # ATTR1 depende de ATTR2: si la relación es 100% consistente, se usa para imputar
    if 'ATTR1' in df.columns and 'ATTR2' in df.columns:
        for valor_attr2 in df['ATTR2'].unique():
            subset = df[(df['ATTR2'] == valor_attr2) & df['ATTR1'].notna()]
            if len(subset) > 0:
                valor_attr1 = subset['ATTR1'].mode()[0]
                if (subset['ATTR1'] == valor_attr1).mean() == 1.0:
                    mascara = df['ATTR1'].isnull() & (df['ATTR2'] == valor_attr2)
                    df.loc[mascara, 'ATTR1'] = valor_attr1
    if 'ATTR3' in df.columns:
        df['ATTR3'] = df['ATTR3'].fillna('No Definido')  # ya existe esta categoría en la tabla
    if 'ITEM' in df.columns:
        df['ITEM'] = df['ITEM'].fillna('Item Desconocido')

    df.drop_duplicates(inplace=True)

    print(f"- {nombre:14} filas {filas_ini:>7,} → {len(df):>7,}  |  "
          f"nulos {nulos_ini:>3} → {df.isnull().sum().sum():>1}  |  "
          f"dup {dup_ini:>3} → {df.duplicated().sum():>1}")
    return df

df_category = limpiar_df(df_category, "DIM_CATEGORY")
df_product = limpiar_df(df_product, "DIM_PRODUCT")
df_segment = limpiar_df(df_segment, "DIM_SEGMENT")
df_calendar = limpiar_df(df_calendar, "DIM_CALENDAR")
df_sales = limpiar_df(df_sales, "FACT_SALES")

print("\n• DIM_CATEGORY tras limpieza (el salto de línea quedó eliminado por .str.strip()):")
print(df_category)

- DIM_CATEGORY   filas       5 →       5  |  nulos   0 → 0  |  dup   0 → 0
- DIM_PRODUCT    filas     505 →     505  |  nulos  14 → 0  |  dup   0 → 0
- DIM_SEGMENT    filas      53 →      52  |  nulos   1 → 0  |  dup   0 → 0
- DIM_CALENDAR   filas     156 →     156  |  nulos   0 → 0  |  dup   0 → 0
- FACT_SALES     filas 122,002 → 122,002  |  nulos   0 → 0  |  dup   0 → 0

• DIM_CATEGORY tras limpieza (el salto de línea quedó eliminado por .str.strip()):
   ID_CATEGORY                    CATEGORY
0            1  Fabric Treatment And Sanit
1            2                    Air Care
2            3                Lavavajillas
3            4            Mega Superficies
4            5         Lavatory Care & Brc


# 4. Integración de las tablas

Se consolidan las cinco tablas en un único DataFrame mediante `LEFT JOIN`s sucesivos, partiendo de `FACT_SALES` para preservar la totalidad de las transacciones:

| Unión | Llave |
|---|---|
| `FACT_SALES` + `DIM_PRODUCT` | `ITEM_CODE` = `ITEM` |
| + `DIM_CATEGORY` | `CATEGORY` = `ID_CATEGORY` |
| + `DIM_SEGMENT` | `CATEGORY` + `ATTR1` + `ATTR2` + `ATTR3` + `FORMAT` (llave compuesta) |
| + `DIM_CALENDAR` | `WEEK` |

In [44]:
def consolidar_dataframes(df_sales, df_product, df_category, df_segment, df_calendar):
    filas_originales = len(df_sales)

    df = pd.merge(df_sales, df_product, left_on='ITEM_CODE', right_on='ITEM',
                   how='left', suffixes=('', '_product'))
    df = pd.merge(df, df_category, left_on='CATEGORY', right_on='ID_CATEGORY',
                   how='left', suffixes=('', '_category'))
    df = pd.merge(df, df_segment, on=['CATEGORY', 'ATTR1', 'ATTR2', 'ATTR3', 'FORMAT'],
                   how='left', suffixes=('', '_segment'))
    df = pd.merge(df, df_calendar, on='WEEK', how='left', suffixes=('', '_calendar'))

    assert len(df) == filas_originales, "El número de ventas cambió durante la consolidación"

    print(f"• Ventas originales conservadas: {filas_originales:,}")
    print(f"• Dimensiones del consolidado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    print(f"• Nulos totales: {df.isnull().sum().sum():,}")
    print(f"• Duplicados: {df.duplicated().sum()}")
    return df

df_consolidado = consolidar_dataframes(df_sales, df_product, df_category, df_segment, df_calendar)

print(f"\n• Información del DataFrame consolidado:")
df_consolidado.info()

print(f"\n• Muestra de 5 filas aleatorias:")
print(df_consolidado.sample(5))

• Ventas originales conservadas: 122,002
• Dimensiones del consolidado: 122,002 filas x 22 columnas
• Nulos totales: 0
• Duplicados: 0

• Información del DataFrame consolidado:
<class 'pandas.DataFrame'>
RangeIndex: 122002 entries, 0 to 122001
Data columns (total 22 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   WEEK                         122002 non-null  str           
 1   ITEM_CODE                    122002 non-null  str           
 2   TOTAL_UNIT_SALES             122002 non-null  float64       
 3   TOTAL_VALUE_SALES            122002 non-null  float64       
 4   TOTAL_UNIT_AVG_WEEKLY_SALES  122002 non-null  float64       
 5   REGION                       122002 non-null  str           
 6   MANUFACTURER                 122002 non-null  str           
 7   BRAND                        122002 non-null  str           
 8   ITEM                         122002 non-null  str         

# 5. Corrección crítica: doble conteo por el consolidado nacional

`REGION` no contiene seis regiones independientes: contiene seis **áreas** (`Area 1`-`Area 6`) más una fila `Total Autos Scanning Mexico`, que es la suma nacional de esas seis áreas, no una región adicional. Si se agrupa por `REGION` y se suma sin filtrar, el total nacional se cuenta dos veces: una al sumar las seis áreas y otra al incluir la fila que ya es su suma.

Se comprueba numéricamente y se corrige agregando una columna `REGION_TYPE` que distingue `Área Regional` de `Total Nacional`, de forma que cualquier análisis posterior (EDA, dashboards, clustering) pueda excluir el total nacional al sumar por región, evitando el error sistemáticamente en vez de recordarlo manualmente cada vez.

In [45]:
ventas_por_region = df_consolidado.groupby('REGION')['TOTAL_VALUE_SALES'].sum().sort_values(ascending=False)
print("• Ventas por valor de REGION (antes de corregir):")
print(ventas_por_region.round(2))

suma_areas = ventas_por_region.drop('Total Autos Scanning Mexico').sum()
total_mexico = ventas_por_region['Total Autos Scanning Mexico']
print(f"\n• Suma de las 6 áreas:      ${suma_areas:,.2f}")
print(f"• Fila 'Scanning Mexico':   ${total_mexico:,.2f}")
print(f"• Diferencia:               ${abs(suma_areas - total_mexico):,.2f} "
      f"({abs(suma_areas - total_mexico) / total_mexico * 100:.4f}% → redondeo, confirma que es la misma cifra)")
print(f"\nSi se suman TODAS las filas de REGION sin filtrar, el total reportado sería "
      f"${ventas_por_region.sum():,.2f} → el doble de las ventas reales (${suma_areas:,.2f})")

• Ventas por valor de REGION (antes de corregir):
REGION
Total Autos Scanning Mexico    5521429.32
Total Autos Area 2             1188796.15
Total Autos Area 5             1153335.54
Total Autos Area 6              983957.57
Total Autos Area 3              803655.34
Total Autos Area 1              714249.98
Total Autos Area 4              677436.00
Name: TOTAL_VALUE_SALES, dtype: float64

• Suma de las 6 áreas:      $5,521,430.57
• Fila 'Scanning Mexico':   $5,521,429.32
• Diferencia:               $1.25 (0.0000% → redondeo, confirma que es la misma cifra)

Si se suman TODAS las filas de REGION sin filtrar, el total reportado sería $11,042,859.89 → el doble de las ventas reales ($5,521,430.57)


In [46]:
# Bandera explícita para separar áreas del consolidado nacional
df_consolidado['REGION_TYPE'] = np.where(
    df_consolidado['REGION'].str.contains('Scanning Mexico', case=False, na=False),
    'Total Nacional',
    'Área Regional')

print(df_consolidado['REGION_TYPE'].value_counts())
print(f"\n• Uso recomendado:")
print(f"  - Análisis por región / suma de todas las áreas → filtrar REGION_TYPE == 'Área Regional'")
print(f"  - Métrica de venta nacional total               → filtrar REGION_TYPE == 'Total Nacional'")
print(f"  - Nunca sumar ambos grupos juntos")

verificacion = df_consolidado[df_consolidado['REGION_TYPE'] == 'Área Regional']['TOTAL_VALUE_SALES'].sum()
nacional = df_consolidado[df_consolidado['REGION_TYPE'] == 'Total Nacional']['TOTAL_VALUE_SALES'].sum()
print(f"\n• Verificación → Suma de áreas: ${verificacion:,.2f}  |  Total nacional: ${nacional:,.2f}")

print(f"\n• Dimensiones nuevas: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

REGION_TYPE
Área Regional     101012
Total Nacional     20990
Name: count, dtype: int64

• Uso recomendado:
  - Análisis por región / suma de todas las áreas → filtrar REGION_TYPE == 'Área Regional'
  - Métrica de venta nacional total               → filtrar REGION_TYPE == 'Total Nacional'
  - Nunca sumar ambos grupos juntos

• Verificación → Suma de áreas: $5,521,430.57  |  Total nacional: $5,521,429.32

• Dimensiones nuevas: 122,002 filas × 23 columnas


# 6. Limpieza estructural del consolidado

Tras los joins pueden quedar columnas con contenido idéntico (por ejemplo, `CATEGORY` original y la que llega de `DIM_CATEGORY`). Se detectan automáticamente comparando el contenido completo de cada columna, y se conserva la versión más descriptiva. También se renombran dos columnas para mayor claridad (`WEEK` → `WEEK_YEAR`, y se resuelve el sufijo redundante de `CATEGORY`).

In [47]:
def eliminar_columnas_duplicadas(df, preferencia):
    '''Detecta columnas con contenido idéntico y conserva solo una, priorizando preferencia.'''
    columnas = df.columns.tolist()
    procesadas, a_eliminar = set(), []

    for i, col1 in enumerate(columnas):
        if col1 in procesadas:
            continue
        iguales = [col1]
        for col2 in columnas[i + 1:]:
            if col2 not in procesadas and df[col1].equals(df[col2]):
                iguales.append(col2)
                procesadas.add(col2)
        if len(iguales) > 1:
            mantener = next((p for p in preferencia if p in iguales), iguales[0])
            eliminar = [c for c in iguales if c != mantener]
            print(f"• Columnas duplicadas: {iguales} → se conserva '{mantener}'")
            a_eliminar.extend(eliminar)

    return df.drop(columns=a_eliminar) if a_eliminar else df

df_consolidado = eliminar_columnas_duplicadas(df_consolidado, preferencia=['ITEM_CODE', 'ID_CATEGORY'])
df_consolidado = df_consolidado.rename(columns={'CATEGORY_category': 'CATEGORY', 'WEEK': 'WEEK_YEAR'})

print(f"\n• Dimensiones nuevas: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

• Columnas duplicadas: ['ITEM_CODE', 'ITEM'] → se conserva 'ITEM_CODE'
• Columnas duplicadas: ['CATEGORY', 'ID_CATEGORY'] → se conserva 'ID_CATEGORY'

• Dimensiones nuevas: 122,002 filas × 21 columnas


### Convención de nombres de columna

De aquí en adelante el resto del pipeline trabaja con nombres de columna en `snake_case` minúscula (`total_value_sales` en vez de `TOTAL_VALUE_SALES`), que es la convención idiomática de Python/pandas y la más común en portafolios de ciencia de datos. Las tablas de origen (`DIM_PRODUCT`, `FACT_SALES`, etc.) se dejaron en mayúsculas tal como llegan del data warehouse, el cambio aplica solo al dataset consolidado que es el entregable final de este proyecto.

In [48]:
df_consolidado.columns = df_consolidado.columns.str.lower()

print("Columnas finales (snake_case):")
print(df_consolidado.columns.tolist())

Columnas finales (snake_case):
['week_year', 'item_code', 'total_unit_sales', 'total_value_sales', 'total_unit_avg_weekly_sales', 'region', 'manufacturer', 'brand', 'item_description', 'format', 'attr1', 'attr2', 'attr3', 'id_category', 'category', 'segment', 'year', 'month', 'week_number', 'date', 'region_type']


# 7. Variables temporales

A partir de `DATE` se derivan componentes de calendario útiles para análisis estacional y series de
tiempo: mes, día de la semana, trimestre, periodo año-mes / año-trimestre y quincena del mes.

In [49]:
df_consolidado['date'] = pd.to_datetime(df_consolidado['date'], errors='coerce')

df_consolidado['month_name'] = df_consolidado['date'].dt.month_name()
df_consolidado['day_number'] = df_consolidado['date'].dt.dayofweek  # 0=Lunes ... 6=Domingo
df_consolidado['day_name'] = df_consolidado['date'].dt.day_name()
df_consolidado['quarter'] = df_consolidado['date'].dt.quarter
df_consolidado['year_month'] = df_consolidado['date'].dt.to_period('M').astype(str)
df_consolidado['year_quarter'] = df_consolidado['year'].astype(str) + '-Q' + df_consolidado['quarter'].astype(str)
df_consolidado['month_period'] = pd.cut(
    df_consolidado['date'].dt.day, bins=[0, 10, 20, 31],
    labels=['Inicio Mes', 'Medio Mes', 'Fin Mes'], include_lowest=True
)

print(f"• Rango de fechas: {df_consolidado['date'].min().date()} a {df_consolidado['date'].max().date()}")
print(f"\n• Muestra aleatoria de las 5 columnas nuevas:")
print(df_consolidado[['date', 'day_number', 'day_name', 'month_name', 'year_month', 'quarter', 'year_quarter', 'month_period']].sample(5, random_state=1))
print(f"\n• Dimensiones nuevas: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

• Rango de fechas: 2022-01-09 a 2023-07-17

• Muestra aleatoria de las 5 columnas nuevas:
            date  day_number day_name month_name year_month  quarter  \
98025 2022-03-20           6   Sunday      March    2022-03        1   
59435 2022-05-01           6   Sunday        May    2022-05        2   
27516 2022-04-10           6   Sunday      April    2022-04        2   
93448 2022-10-16           6   Sunday    October    2022-10        4   
81511 2023-05-29           0   Monday        May    2023-05        2   

      year_quarter month_period  
98025      2022-Q1    Medio Mes  
59435      2022-Q2   Inicio Mes  
27516      2022-Q2   Inicio Mes  
93448      2022-Q4    Medio Mes  
81511      2023-Q2      Fin Mes  

• Dimensiones nuevas: 122,002 filas × 28 columnas


# 8. Métricas de venta derivadas

Se calcula el precio promedio de venta semanal (ASP, *Average Selling Price*) y dos medidas de desviación respecto al promedio semanal histórico del producto, que permiten identificar semanas atípicas de venta.

**Nota sobre `avg_weekly_price`**: tanto `total_value_sales` como `total_unit_sales` ya son totales agregados de la semana (no vienen de una transacción individual), así que `avg_weekly_price = total_value_sales / total_unit_sales` no es un precio de lista fijo, es el **precio promedio al que se vendió el producto esa semana**, y puede variar de una semana a otra por efecto de promociones o mezcla de canales. Es la forma estándar de derivar precio cuando solo se cuenta con datos agregados (sin nivel transacción).

In [50]:
df_consolidado['avg_weekly_price'] = (
    df_consolidado['total_value_sales'] / df_consolidado['total_unit_sales']
).replace([np.inf, -np.inf], np.nan).round(2)

df_consolidado['var_weekly_avg'] = (
    df_consolidado['total_unit_sales'] - df_consolidado['total_unit_avg_weekly_sales']
).round(2)

df_consolidado['var_pct'] = (
    (df_consolidado['total_unit_sales'] / df_consolidado['total_unit_avg_weekly_sales'] - 1) * 100
).replace([np.inf, -np.inf], np.nan).round(2)

df_consolidado['above_avg'] = (
    df_consolidado['total_unit_sales'] > df_consolidado['total_unit_avg_weekly_sales']
).astype(int)

print("• Cuatro columnas nuevas:")
print(f"- avg_weekly_price → rango ${df_consolidado['avg_weekly_price'].min():.2f} - ${df_consolidado['avg_weekly_price'].max():.2f}, "
      f"promedio ${df_consolidado['avg_weekly_price'].mean():.2f}")
print(f"- var_weekly_avg   → mediana {df_consolidado['var_weekly_avg'].median():.2f} uds, "
      f"{(df_consolidado['var_weekly_avg'] < 0).mean()*100:.1f}% de las semanas quedan por debajo de su propio promedio histórico")
print(f"- var_pct          → mediana {df_consolidado['var_pct'].median():.1f}%, "
      f"{(df_consolidado['var_pct'].abs() > 100).mean()*100:.2f}% de las filas con variación extrema (>100% respecto a su historial)")
print(f"- above_avg        → {df_consolidado['above_avg'].mean()*100:.1f}% de las ventas superan su promedio semanal")
print(f"\n• Dimensiones nuevas: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

• Cuatro columnas nuevas:
- avg_weekly_price → rango $0.55 - $298.30, promedio $55.12
- var_weekly_avg   → mediana -3.10 uds, 96.5% de las semanas quedan por debajo de su propio promedio histórico
- var_pct          → mediana -91.8%, 0.99% de las filas con variación extrema (>100% respecto a su historial)
- above_avg        → 3.5% de las ventas superan su promedio semanal

• Dimensiones nuevas: 122,002 filas × 32 columnas


# 9. Categorización de ventas

Tres variables categóricas (por monto, por unidades y por precio) que facilitan segmentar y visualizar el desempeño sin depender de valores continuos.

In [51]:
df_consolidado['cat_sales'] = pd.cut(
    df_consolidado['total_value_sales'], bins=[0, 10, 50, 100, 500, float('inf')],
    labels=['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Muy Alto'], include_lowest=True
)
df_consolidado['cat_units'] = pd.cut(
    df_consolidado['total_unit_sales'], bins=[0, 0.5, 2, 5, 10, float('inf')],
    labels=['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Muy Alto'], include_lowest=True
)
df_consolidado['cat_avg_price'] = pd.cut(
    df_consolidado['avg_weekly_price'], bins=[0, 20, 50, 100, 200, float('inf')],
    labels=['Económico', 'Bajo', 'Medio', 'Alto', 'Premium'], include_lowest=True
)

print(f"\n• Contenido de las 3 nuevas columnas categóricas:")
for col in ['cat_sales', 'cat_units', 'cat_avg_price']:
    print()
    print(df_consolidado[col].value_counts().sort_index())

print(f"\n• Dimensiones nuevas: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")


• Contenido de las 3 nuevas columnas categóricas:

cat_sales
Muy Bajo    50399
Bajo        35943
Medio       14776
Alto        16551
Muy Alto     4333
Name: count, dtype: int64

cat_units
Muy Bajo    67777
Bajo        27934
Medio       12051
Alto         6694
Muy Alto     7546
Name: count, dtype: int64

cat_avg_price
Económico    27764
Bajo         47420
Medio        28360
Alto         15313
Premium       3067
Name: count, dtype: int64

• Dimensiones nuevas: 122,002 filas × 35 columnas


# 10. Región legible y tamaño del producto

Se generan versiones legibles de `region` para reportes (`region_clean`, `region_short`), removiendo "Total Autos Scanning" y sin alterar la bandera `region_type` creada en la sección 5, que sigue siendo la fuente para evitar el doble conteo. 

Se extrae el tamaño del producto (número + unidad estandarizada) desde `item_description` mediante expresiones regulares.

In [52]:
df_consolidado['region_clean'] = df_consolidado['region'].str.replace('Total Autos ', '', regex=False).str.title()
df_consolidado['region_short'] = df_consolidado['region_clean'].str.replace('Scanning ', '', regex=False)

df_consolidado['size'] = (
    df_consolidado['item_description']
    .str.extract(r'(\d+\.?\d*\s*(?:ML|L|LT|G|KG|GR|Gr|Grs|Kg|Lt|Lts|Ml|M))', expand=False)
    .str.replace(r'\s+', '', regex=True)
)
unidades = {
    r'(\d+\.?\d*)ML$': r'\1ml', r'(\d+\.?\d*)Ml$': r'\1ml', r'(\d+\.?\d*)mL$': r'\1ml', r'(\d+\.?\d*)M$': r'\1ml',
    r'(\d+\.?\d*)LT$': r'\1l', r'(\d+\.?\d*)Lt$': r'\1l', r'(\d+\.?\d*)LTS$': r'\1l', r'(\d+\.?\d*)Lts$': r'\1l', r'(\d+\.?\d*)L$': r'\1l',
    r'(\d+\.?\d*)GR$': r'\1g', r'(\d+\.?\d*)Gr$': r'\1g', r'(\d+\.?\d*)GRS$': r'\1g', r'(\d+\.?\d*)Grs$': r'\1g', r'(\d+\.?\d*)G$': r'\1g',
    r'(\d+\.?\d*)KG$': r'\1kg', r'(\d+\.?\d*)Kg$': r'\1kg', r'(\d+\.?\d*)K$': r'\1kg',
}
for patron, reemplazo in unidades.items():
    df_consolidado['size'] = df_consolidado['size'].str.replace(patron, reemplazo, regex=True)

df_consolidado['size_num'] = pd.to_numeric(df_consolidado['size'].str.extract(r'(\d+\.?\d*)', expand=False), errors='coerce')
df_consolidado['size_unit'] = df_consolidado['size'].str.extract(r'([a-z]+)$', expand=False)

print("• Regiones legibles:", df_consolidado['region_short'].unique().tolist())
print(f"\n• size sin patrón reconocible (por tratar): {df_consolidado['size'].isnull().sum():,} filas "
      f"({df_consolidado['size'].isnull().mean()*100:.1f}%)")
print(f"\n• Muestra aleatoria de las 3 columnas nuevas:")
print(df_consolidado[['item_description', 'size', 'size_num', 'size_unit']].sample(5, random_state=1))
print(f"\n• Dimensiones nuevas: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

• Regiones legibles: ['Area 5', 'Area 6', 'Mexico', 'Area 3', 'Area 2', 'Area 1', 'Area 4']

• size sin patrón reconocible (por tratar): 1,200 filas (1.0%)

• Muestra aleatoria de las 3 columnas nuevas:
                                        item_description    size  size_num  \
98025  Dr Beckmann El Mago Quitamanchas 8 Liquido Bot...      8l     8.000   
59435  Vanish White Removerd D/Manchas Y Blanq.S/Clor...  4000ml  4000.000   
27516  Vanish Gel Crystal White Bot 4Lt + Mm Rosa Pou...      4l     4.000   
93448  Clarasol Regular Sin Aroma Garrafon 3.785 Lt 0...  3.785l     3.785   
81511  Clorox Blancos Brillantes Quitamanchas Pouch 7...   730ml   730.000   

      size_unit  
98025         l  
59435        ml  
27516         l  
93448         l  
81511        ml  

• Dimensiones nuevas: 122,002 filas × 40 columnas


# 11. Indicadores de negocio

Cuatro banderas binarias basadas en la mediana de precio, rotación y valor de venta (más robusta ante outliers que el promedio), que identifican productos y transacciones de alto desempeño.

La columna `total_unit_avg_weekly_sales` ya es un promedio móvil (varía semana a semana para el mismo producto). Tomar la mediana directamente sobre todas las filas le da más peso a los productos con más semanas/regiones registradas, no a los que realmente tienen mayor rotación. Se colapsa primero a un valor por producto (promedio de su propio promedio semanal) para que cada producto cuente una sola vez en el benchmark.

In [53]:
mediana_precio = df_consolidado['avg_weekly_price'].median()
mediana_valor = df_consolidado['total_value_sales'].median()

rotacion_por_producto = df_consolidado.groupby('item_code')['total_unit_avg_weekly_sales'].mean()
mediana_rotacion = rotacion_por_producto.median()

df_consolidado['high_value'] = (df_consolidado['avg_weekly_price'] > mediana_precio).astype(int)
df_consolidado['high_turnover'] = (df_consolidado['total_unit_avg_weekly_sales'] > mediana_rotacion).astype(int)
df_consolidado['vip_sale'] = (df_consolidado['total_value_sales'] > mediana_valor).astype(int)
df_consolidado['star_product'] = ((df_consolidado['high_value'] == 1) & (df_consolidado['high_turnover'] == 1)).astype(int)

print(f"• Medianas → precio: ${mediana_precio:.2f} | rotación (por producto): {mediana_rotacion:.2f} uds/semana | valor: ${mediana_valor:.2f}")
print(f"\n• Cuatro columnas nuevas:")
n_high_value = df_consolidado.loc[df_consolidado['high_value'] == 1, 'item_code'].nunique()
print(f"- high_value    → {df_consolidado['high_value'].mean()*100:.1f}% de las transacciones ({n_high_value} productos únicos con precio sobre la mediana)")

n_high_turnover = df_consolidado.loc[df_consolidado['high_turnover'] == 1, 'item_code'].nunique()
print(f"- high_turnover → {df_consolidado['high_turnover'].mean()*100:.1f}% de las transacciones ({n_high_turnover} productos únicos con rotación sobre la mediana)")

pct_valor_vip = df_consolidado.loc[df_consolidado['vip_sale'] == 1, 'total_value_sales'].sum() / df_consolidado['total_value_sales'].sum() * 100
print(f"- vip_sale      → {df_consolidado['vip_sale'].mean()*100:.1f}% de las transacciones, pero concentran {pct_valor_vip:.1f}% del valor total de venta")

n_star = df_consolidado.loc[df_consolidado['star_product'] == 1, 'item_code'].nunique()
print(f"- star_product  → {df_consolidado['star_product'].mean()*100:.1f}% de las ventas (alto precio + alta rotación simultáneos, {n_star} productos únicos)")
print(f"\n• Dimensiones nuevas: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

• Medianas → precio: $36.64 | rotación (por producto): 3.20 uds/semana | valor: $16.81

• Cuatro columnas nuevas:
- high_value    → 50.0% de las transacciones (184 productos únicos con precio sobre la mediana)
- high_turnover → 59.0% de las transacciones (281 productos únicos con rotación sobre la mediana)
- vip_sale      → 50.0% de las transacciones, pero concentran 97.5% del valor total de venta
- star_product  → 18.8% de las ventas (alto precio + alta rotación simultáneos, 150 productos únicos)

• Dimensiones nuevas: 122,002 filas × 44 columnas


# 12. Rankings y percentiles

Tres rankings a distinto nivel de granularidad: producto histórico, posición dentro de cada región (calculado sobre `region` sin filtrar `region_type`, ya que aquí interesa la posición relativa de cada venta dentro de su propia fila de región, no una suma) y posición del producto dentro de cada año. Se ocupa `method='dense'` donde los empates comparten el mismo rank (ej. dos productos empatados en 1er lugar reciben ambos rank=1), y el siguiente valor distinto continúa en el número consecutivo (2, no 3).

In [54]:
df_consolidado['rank_product_value'] = (
    df_consolidado.groupby('item_code')['total_value_sales'].transform('sum')
    .rank(ascending=False, method='dense')
)
df_consolidado['percentile_region'] = df_consolidado.groupby('region')['total_value_sales'].transform(
    lambda x: pd.qcut(x, q=10, labels=False, duplicates='drop')
)
df_consolidado['rank_year_item'] = df_consolidado.groupby(['year', 'item_code'])['total_value_sales'].rank(ascending=False, method='dense')

print(f"• Tres columnas nuevas:")
print(f"- rank_product_value → 1 (más vendido) a {int(df_consolidado['rank_product_value'].max())} (menos vendido), "
      f"{df_consolidado['rank_product_value'].nunique()} productos únicos")
print(f"- percentile_region  → deciles 0 (10% inferior) a 9 (10% superior) dentro de cada región")

n_grupos = df_consolidado.groupby(['year', 'item_code']).ngroups
n_rank1 = (df_consolidado['rank_year_item'] == 1).sum()
print(f"- rank_year_item     → identifica la mejor semana-región de cada producto dentro de cada año ({n_grupos} combinaciones producto-año)")
print(f"                       {n_rank1} de 122,002 filas con rank=1.0")
print(f"\n• Dimensiones nuevas: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

• Tres columnas nuevas:
- rank_product_value → 1 (más vendido) a 343 (menos vendido), 343 productos únicos
- percentile_region  → deciles 0 (10% inferior) a 9 (10% superior) dentro de cada región
- rank_year_item     → identifica la mejor semana-región de cada producto dentro de cada año (637 combinaciones producto-año)
                       816 de 122,002 filas con rank=1.0

• Dimensiones nuevas: 122,002 filas × 47 columnas


# 13. Organización final y tratamiento de nulos remanentes

Se reordenan las columnas en grupos lógicos para facilitar la lectura del dataset, y se tratan los únicos nulos que quedan tras las transformaciones: `size`/`size_num`/`size_unit` (productos cuya descripción no sigue el patrón estándar de tamaño, p. ej. cajas o kits) y `avg_weekly_price`/`cat_avg_price` (filas con 0 unidades vendidas, donde el precio promedio de venta no es calculable).

In [55]:
columnas_ordenadas = [
    'item_code', 'manufacturer', 'brand', 'item_description',
    'id_category', 'category', 'segment', 'format', 'attr1', 'attr2', 'attr3',
    'size', 'size_num', 'size_unit',
    'date', 'day_number', 'day_name', 'week_number', 'week_year', 'month', 'month_name',
    'quarter', 'year_month', 'year_quarter', 'month_period', 'year',
    'region', 'region_type', 'region_clean', 'region_short',
    'total_unit_sales', 'total_value_sales', 'total_unit_avg_weekly_sales', 'avg_weekly_price', 'var_weekly_avg', 'var_pct',
    'cat_sales', 'cat_units', 'cat_avg_price',
    'above_avg', 'high_value', 'high_turnover', 'vip_sale', 'star_product',
    'rank_product_value', 'percentile_region', 'rank_year_item',
]
df_consolidado = df_consolidado[columnas_ordenadas]
print(f"Dataset reorganizado: {df_consolidado.shape[0]:,} filas x {df_consolidado.shape[1]} columnas")

Dataset reorganizado: 122,002 filas x 47 columnas


In [56]:
# size: si item_description referencia el tamaño estándar de 3.785 L, rellenar explícitamente
mascara_3785 = df_consolidado['size'].isnull() & df_consolidado['item_description'].str.contains('3.785', na=False)
df_consolidado.loc[mascara_3785, ['size', 'size_num', 'size_unit']] = ['3.785l', 3.785, 'l']

mascara_resto = df_consolidado['size'].isnull()
df_consolidado.loc[mascara_resto, ['size', 'size_num', 'size_unit']] = ['Sin Especificar', 0, 'Sin Especificar']

print(f"• size → {mascara_3785.sum()} filas rellenadas como '3.785l', {mascara_resto.sum()} como 'Sin Especificar'")

# avg_weekly_price: 0 unidades vendidas → precio no calculable, se define como 0
n_price_nulo = df_consolidado['avg_weekly_price'].isnull().sum()
df_consolidado['avg_weekly_price'] = df_consolidado['avg_weekly_price'].fillna(0)

df_consolidado['cat_avg_price'] = pd.cut(
    df_consolidado['avg_weekly_price'], bins=[0, 20, 50, 100, 200, float('inf')],
    labels=['Económico', 'Bajo', 'Medio', 'Alto', 'Premium'], include_lowest=False
)
df_consolidado['cat_avg_price'] = df_consolidado['cat_avg_price'].cat.add_categories(['Sin Especificar'])
df_consolidado.loc[df_consolidado['avg_weekly_price'] == 0, 'cat_avg_price'] = 'Sin Especificar'

print(f"• avg_weekly_price → {n_price_nulo} filas rellenadas con 0 (ventas con 0 unidades)")
print(f"\n• Nulos totales tras el tratamiento: {df_consolidado.isnull().sum().sum()}")

• size → 112 filas rellenadas como '3.785l', 1088 como 'Sin Especificar'
• avg_weekly_price → 78 filas rellenadas con 0 (ventas con 0 unidades)

• Nulos totales tras el tratamiento: 0


# 14. Validación final

Verificación de que el dataset consolidado no contiene nulos, duplicados ni columnas con tipo de
dato incorrecto, y que las cifras de venta cuadran tras la corrección de la sección 5.

In [57]:
print(f"• Duplicados: {df_consolidado.duplicated().sum()}")
print(f"• Nulos: {df_consolidado.isnull().sum().sum()}")
print(f"• Dimensiones finales: {df_consolidado.shape[0]:,} filas x {df_consolidado.shape[1]} columnas")

check = df_consolidado.groupby('region_type')['total_value_sales'].sum()
print(f"\n• Control de doble conteo:")
print(check.round(2))
print(f"- Diferencia relativa: {abs(check['Área Regional'] - check['Total Nacional']) / check['Total Nacional'] * 100:.4f}%")

print(f"\n• Información del DataFrame consolidado:")
print(df_consolidado.info())

• Duplicados: 0
• Nulos: 0
• Dimensiones finales: 122,002 filas x 47 columnas

• Control de doble conteo:
region_type
Total Nacional    5521429.32
Área Regional     5521430.57
Name: total_value_sales, dtype: float64
- Diferencia relativa: 0.0000%

• Información del DataFrame consolidado:
<class 'pandas.DataFrame'>
RangeIndex: 122002 entries, 0 to 122001
Data columns (total 47 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   item_code                    122002 non-null  str           
 1   manufacturer                 122002 non-null  str           
 2   brand                        122002 non-null  str           
 3   item_description             122002 non-null  str           
 4   id_category                  122002 non-null  int64         
 5   category                     122002 non-null  str           
 6   segment                      122002 non-null  str           
 7   format      

### Diccionario de columnas

Referencia sobre qué contiene cada una de las 47 columnas del dataset consolidado.

In [58]:
diccionario_columnas = {
    'item_code': 'Código identificador único del producto (SKU). Llave usada para unir FACT_SALES con DIM_PRODUCT.',
    'manufacturer': 'Fabricante o compañía dueña de la marca.',
    'brand': 'Marca comercial del producto.',
    'item_description': 'Descripción completa del producto tal como aparece en el catálogo (nombre, variante, tamaño, código de barras).',
    'id_category': 'Identificador numérico de la categoría de producto, tal como llega de DIM_CATEGORY.',
    'category': 'Nombre de la categoría de producto (agrupación más amplia que segment).',
    'segment': 'Segmento de producto dentro de la categoría.',
    'format': 'Formato o presentación física del producto.',
    'attr1': 'Atributo de producto de nivel 1 (subcategoría funcional), depende de attr2.',
    'attr2': 'Atributo de producto de nivel 2.',
    'attr3': 'Atributo de producto de nivel 3 (más específico, ej. color o variante).',
    'size': 'Tamaño del producto extraído de item_description, con unidad estandarizada (ml, l, g, kg).',
    'size_num': 'Parte numérica de size, para cálculos y ordenamientos.',
    'size_unit': 'Unidad de medida de size.',
    'date': 'Fecha de inicio de la semana comercial de la venta.',
    'day_number': 'Día de la semana de date en formato numérico (0=lunes ... 6=domingo).',
    'day_name': 'Nombre del día de la semana de date, en inglés (salida nativa de pandas).',
    'week_number': 'Número de semana comercial dentro del año.',
    'week_year': 'Semana y año combinados, formato original del calendario comercial.',
    'month': 'Número de mes (1-12) de date.',
    'month_name': 'Nombre del mes de date, en inglés.',
    'quarter': 'Trimestre del año (1-4) de date.',
    'year_month': 'Año y mes combinados, formato AAAA-MM. Útil para series de tiempo mensuales.',
    'year_quarter': 'Año y trimestre combinados, formato AAAA-QN.',
    'month_period': 'Parte del mes en que cae la fecha, según el día del mes.',
    'year': 'Año de la venta.',
    'region': 'Región de venta tal como llega de la fuente original, incluye tanto áreas como el consolidado nacional.',
    'region_type': 'Bandera creada en el ETL para distinguir área regional de consolidado nacional.',
    'region_clean': 'Versión legible de region, sin el prefijo \'Total Autos\'.',
    'region_short': 'Versión corta de region_clean, sin la palabra \'Scanning\' en el caso del total nacional.',
    'total_unit_sales': 'Unidades vendidas en la semana para ese producto y región.',
    'total_value_sales': 'Valor monetario de la venta en la semana, en pesos mexicanos ($).',
    'total_unit_avg_weekly_sales': 'Promedio histórico de unidades vendidas por semana para ese producto (dato de origen).',
    'avg_weekly_price': 'Precio promedio de venta semanal (ASP, Average Selling Price), calculado como total_value_sales / total_unit_sales.',
    'var_weekly_avg': 'Diferencia entre las unidades vendidas esa semana y el promedio histórico semanal del producto.',
    'var_pct': 'Igual que var_weekly_avg pero expresado en porcentaje respecto al promedio histórico.',
    'cat_sales': 'Categorización de total_value_sales en rangos (Muy Bajo a Muy Alto).',
    'cat_units': 'Categorización de total_unit_sales en rangos (Muy Bajo a Muy Alto).',
    'cat_avg_price': 'Categorización de avg_weekly_price en rangos (Económico a Premium).',
    'above_avg': '1 si la venta de la semana superó el promedio histórico semanal del producto (total_unit_avg_weekly_sales).',
    'high_value': '1 si avg_weekly_price está por encima de la mediana de precio de todo el dataset.',
    'high_turnover': '1 si la rotación promedio del producto (total_unit_avg_weekly_sales) está por encima de la mediana de rotación entre productos.',
    'vip_sale': '1 si total_value_sales está por encima de la mediana de valor de venta de todo el dataset.',
    'star_product': '1 si la venta cumple high_value=1 y high_turnover=1 simultáneamente (alto precio y alta rotación).',
    'rank_product_value': 'Posición del producto (item_code) según su venta total acumulada en todo el periodo. 1 = producto más vendido.',
    'percentile_region': 'Decil de total_value_sales de la fila dentro de su propia region (0=10% inferior, 9=10% superior).',
    'rank_year_item': 'Posición de la venta dentro de su mismo producto (item_code) y año (year), ordenado por total_value_sales.',
}

df_diccionario = pd.DataFrame(list(diccionario_columnas.items()), columns=['columna', 'descripcion'])
print(f"• {len(df_diccionario)} columnas documentadas:\n")
for _, fila in df_diccionario.iterrows():
    print(f" - {fila['columna']:28} {fila['descripcion']}")

• 47 columnas documentadas:

 - item_code                    Código identificador único del producto (SKU). Llave usada para unir FACT_SALES con DIM_PRODUCT.
 - manufacturer                 Fabricante o compañía dueña de la marca.
 - brand                        Marca comercial del producto.
 - item_description             Descripción completa del producto tal como aparece en el catálogo (nombre, variante, tamaño, código de barras).
 - id_category                  Identificador numérico de la categoría de producto, tal como llega de DIM_CATEGORY.
 - category                     Nombre de la categoría de producto (agrupación más amplia que segment).
 - segment                      Segmento de producto dentro de la categoría.
 - format                       Formato o presentación física del producto.
 - attr1                        Atributo de producto de nivel 1 (subcategoría funcional), depende de attr2.
 - attr2                        Atributo de producto de nivel 2.
 - attr3         

# 15. Exportación del dataset consolidado

Se exporta en tres formatos para distintos casos de uso: CSV (portabilidad), Excel con hojas de
estadísticas e info de columnas (revisión manual / stakeholders no técnicos) y Parquet (lectura
eficiente en herramientas de análisis y modelado posteriores).

In [59]:
# CSV
df_consolidado.to_csv('df_consolidado_final.csv', index=False, encoding='utf-8-sig')

# Excel con múltiples hojas
with pd.ExcelWriter('df_consolidado_final.xlsx', engine='openpyxl') as writer:
    df_consolidado.to_excel(writer, sheet_name='Datos Consolidados', index=False) # Datos completos
    df_consolidado.describe(include='all').T.to_excel(writer, sheet_name='Estadísticas') # Resumen estadístico
    pd.DataFrame({
        'Columna': df_consolidado.columns,
        'Tipo': df_consolidado.dtypes.astype(str).values,
        'Nulos': df_consolidado.isnull().sum().values,
        'Únicos': df_consolidado.nunique().values
    }).to_excel(writer, sheet_name='Info Columnas', index=False) # Info de columnas

# Parquet
df_consolidado.to_parquet('df_consolidado_final.parquet', index=False)

print("Archivos exportados: df_consolidado_final.csv / .xlsx / .parquet")

Archivos exportados: df_consolidado_final.csv / .xlsx / .parquet


# Conclusiones

**Resultado del pipeline:**

  Un dataset consolidado para análisis de 122,002 ventas semanales × 47 columnas, sin nulos ni duplicados, que integra información de producto, categoría, segmento y calendario sobre la tabla de hechos original. Las columnas finales usan `snake_case` minúscula (convención idiomática de Python/pandas); las tablas de origen se dejaron tal como llegan del data warehouse.

**Hallazgos y decisiones clave:**

  - El catálogo de productos contenía un typo de captura (prefijo "9" espurio) en 11 descripciones, corregido puntualmente tras confirmar que el patrón no aparecía en ningún registro legítimo.
  - La región `Total Autos Scanning Mexico` no es una región más: es el consolidado nacional de las seis áreas. Sumarla junto con las áreas duplica las cifras de venta, un error fácil de cometer al hacer un `groupby('region')` directo. Se resolvió con una bandera explícita (`region_type`)
  para que cualquier análisis posterior separe correctamente "por área" de "total nacional".
  - `avg_weekly_price` se nombró para dejar explícito que es un precio promedio de venta semanal derivado de totales agregados, no un precio de lista fijo por unidad.
  - `high_turnover` se calcula sobre la mediana de rotación colapsada por producto, no sobre todas las filas del dataset, para no darle más peso a los productos con más semanas/regiones registradas.
  - Los nulos remanentes tras la consolidación (tamaño de producto no estandarizado, precio no calculable por 0 unidades vendidas) se trataron con reglas explícitas y documentadas, en vez de eliminarse, para no perder información de venta real.

**Próximos pasos:**

  Este dataset consolidado es la base de dos líneas de trabajo independientes del portafolio: por un lado, el análisis exploratorio (EDA) y el modelo de segmentación (K-Means), que comparten el mismo criterio de limpieza de principio a fin; y por otro, un proyecto de predicción de ventas con regresión lineal múltiple y series de tiempo (ARIMA), que parte de este mismo dataset consolidado con su propio análisis exploratorio.